# Task 2 — class-level errors and probability decisions

This notebook reads the current `eda_original_v2` dataset from its JSON card. It reuses the saved, fitted CatBoost baseline and the existing stratified 80/20 split, without importing any notebook variables. Five development folds supply out-of-fold probabilities for a fixed class-bias decision rule. The existing holdout checks the selected rule once. All preprocessing is fitted within each training fold.

In [1]:
from pathlib import Path
from hashlib import sha256
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents)
          if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
import numpy as np,pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import f1_score,classification_report,confusion_matrix
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task2.models.decision_rules import ClassBiasClassifier,restore_categorical_nulls
SEED=set_seed()
DATA_NAME='eda_original_v2';TARGET='segment';ID='player_id'
data_card=json.loads((SRC/'task2'/'data'/'eda'/f'{DATA_NAME}.json').read_text(encoding='utf8'))
assert data_card['name']==DATA_NAME and data_card['task']=='task2'
train_path=SRC/data_card['splits']['train']['path']
train=pd.read_parquet(train_path) if data_card['format']=='parquet' else pd.read_csv(train_path)
assert train[ID].is_unique and train[TARGET].notna().all()
features=[col for col in train if col not in (ID,TARGET)]
y=train[TARGET].astype(int)
assert set(y)=={0,1,2,3}
raw_X=train[features]
categorical=[col for col in raw_X.select_dtypes(include=['object','str']).columns]
assert len(categorical)==10
X=restore_categorical_nulls(raw_X,categorical)
dev_idx,valid_idx=train_test_split(np.arange(len(y)),test_size=.20,
    stratify=y,random_state=SEED)
X_dev,X_valid=X.iloc[dev_idx],X.iloc[valid_idx]
y_dev,y_valid=y.iloc[dev_idx],y.iloc[valid_idx]
assert y_dev.index.intersection(y_valid.index).empty
model_name='t2_cat_original_no_noise_v2'
model_card=json.loads((SRC/'task2'/'save'/f'{model_name}.json').read_text(encoding='utf8'))
assert model_card['data_name']==DATA_NAME
baseline=joblib.load(SRC/model_card['model_path'])
classes=np.asarray(baseline.named_steps['classifier'].classes_)
assert classes.tolist()==[0,1,2,3]
baseline_pred=np.asarray(baseline.predict(X_valid)).reshape(-1)
baseline_score=f1_score(y_valid,baseline_pred,average='macro')
saved_eval=json.loads((SRC/'task2'/'save'/f'{model_name}-evaluation.json').read_text(encoding='utf8'))
assert abs(baseline_score-saved_eval['score'])<1e-10
print('Baseline holdout macro F1:',baseline_score)
print(classification_report(y_valid,baseline_pred,digits=4,zero_division=0))
print('Confusion matrix:');display(pd.DataFrame(confusion_matrix(y_valid,baseline_pred),
    index=['true_casual','true_grinder','true_social','true_whale'],
    columns=['pred_casual','pred_grinder','pred_social','pred_whale']))

Baseline holdout macro F1: 0.7504144211991189
              precision    recall  f1-score   support

           0     0.7802    0.8595    0.8179      8191
           1     0.7481    0.7427    0.7454      5207
           2     0.7423    0.6679    0.7031      4206
           3     0.7836    0.6924    0.7352      3196

    accuracy                         0.7658     20800
   macro avg     0.7636    0.7406    0.7504     20800
weighted avg     0.7651    0.7658    0.7638     20800

Confusion matrix:


,pred_casual,pred_grinder,pred_social,pred_whale
true_casual,7040,710,398,43
true_grinder,1006,3867,122,212
true_social,844,197,2809,356
true_whale,133,395,455,2213


## 1. Paired development-fold probabilities

Clone the saved CatBoost pipeline so each fold refits its imputer, category handling, and classifier on fold training rows only. Cache probabilities and verify fold scores against the previously recorded five-fold baseline.

In [2]:
RESULTS=SRC/'task2'/'models'/'results'
RESULTS.mkdir(parents=True,exist_ok=True)
cv_X=X_dev.reset_index(drop=True);cv_y=y_dev.reset_index(drop=True)
splits=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(cv_X,cv_y))
split_signature=sha256(np.asarray(dev_idx,dtype=np.int64).tobytes()+
    np.asarray(y_dev,dtype=np.int8).tobytes()).hexdigest()
cache_key={'data_name':DATA_NAME,'data_created_at':data_card['created_at'],
    'model_name':model_name,'model_created_at':model_card['created_at'],
    'seed':SEED,'folds':5,'split_signature':split_signature}
parts=[]
for fold,(fit_pos,check_pos) in enumerate(splits,start=1):
    path=RESULTS/f'class_bias_fold{fold}_v1.parquet'
    meta=RESULTS/f'class_bias_fold{fold}_v1.json'
    if path.exists() and meta.exists():
        assert json.loads(meta.read_text(encoding='utf8'))=={'key':cache_key,'fold':fold}
        part=pd.read_parquet(path)
        assert part.row_pos.tolist()==check_pos.tolist()
        print('Loaded probability fold',fold)
    else:
        assert not (path.exists() or meta.exists()),'Inspect incomplete probability cache'
        fitted=Pipeline([
            ("preprocess",clone(baseline.named_steps["preprocess"])),
            ("classifier",CatBoostClassifier(**baseline.named_steps["classifier"].get_params())),
        ])
        fitted.fit(cv_X.iloc[fit_pos],cv_y.iloc[fit_pos])
        probabilities=np.asarray(fitted.predict_proba(cv_X.iloc[check_pos]),dtype=float)
        assert probabilities.shape==(len(check_pos),4) and np.isfinite(probabilities).all()
        part=pd.DataFrame({'fold':fold,'row_pos':check_pos,'true':cv_y.iloc[check_pos].to_numpy(),
            **{f'p_{label}':probabilities[:,label] for label in range(4)}})
        part.to_parquet(path,index=False)
        meta.write_text(json.dumps({'key':cache_key,'fold':fold},indent=2)+'\n',encoding='utf8')
        print('Trained probability fold',fold)
    parts.append(part)
oof=pd.concat(parts,ignore_index=True)
assert oof.row_pos.is_unique and len(oof)==len(cv_X)
pcols=[f'p_{label}' for label in range(4)]
assert np.allclose(oof[pcols].sum(axis=1),1,atol=1e-5)
probabilities=oof[pcols].to_numpy()
labels=oof.true.to_numpy(dtype=int)
folds=oof.fold.to_numpy(dtype=int)
raw_oof_pred=classes[probabilities.argmax(axis=1)]
fold_baseline=pd.DataFrame([
    {'fold':fold,'macro_F1':f1_score(labels[folds==fold],raw_oof_pred[folds==fold],average='macro')}
    for fold in range(1,6)])
display(fold_baseline)
print('OOF baseline macro F1:',f1_score(labels,raw_oof_pred,average='macro'))
old_cv_path=SRC/'task2'/'models'/'round4_cv_v2.json'
if old_cv_path.exists():
    old_cv=json.loads(old_cv_path.read_text(encoding='utf8'))
    assert old_cv['config']['split_signature']==split_signature
    old_scores=np.array([row['cat_full']['macro_F1'] for row in old_cv['folds']])
    print('Previous CatBoost fold scores:',old_scores.round(5))
    print('New OOF fold score difference:',(fold_baseline.macro_F1.to_numpy()-old_scores).round(5))

Loaded probability fold 1
Loaded probability fold 2
Loaded probability fold 3
Loaded probability fold 4
Loaded probability fold 5


,fold,macro_F1
0,1,0.756123
1,2,0.746062
2,3,0.750865
3,4,0.758046
4,5,0.751726


OOF baseline macro F1: 0.7525700687982839
Previous CatBoost fold scores: [0.75612 0.74606 0.75086 0.75805 0.75173]
New OOF fold score difference: [0. 0. 0. 0. 0.]


## 2. Select class biases without touching the holdout

Class 0 is the reference bias (zero); search small log-probability offsets for classes 1–3. Report the optimistic same-OOF tuned score and a leave-one-fold-out bias stability check separately. The unchanged original holdout is the main comparison with the saved baseline.

In [3]:
def predict_with_bias(probability,bias):
    return classes[np.argmax(np.log(np.clip(probability,1e-12,1))+bias,axis=1)]

def fit_biases(probability,actual):
    logp=np.log(np.clip(probability,1e-12,1))
    bias=np.zeros(4,dtype=float)
    grid=np.round(np.arange(-.50,.501,.05),2)
    for _ in range(3):
        changed=False
        for label in (1,2,3):
            best=bias[label];best_score=-1.0
            for candidate in grid:
                trial=bias.copy();trial[label]=candidate
                pred=classes[np.argmax(logp+trial,axis=1)]
                score=f1_score(actual,pred,average='macro',zero_division=0)
                if score>best_score+1e-10 or (abs(score-best_score)<=1e-10 and abs(candidate)<abs(best)):
                    best_score,best=float(score),float(candidate)
            if best!=bias[label]:changed=True
            bias[label]=best
        if not changed:break
    return bias

meta_rows=[]
for fold in range(1,6):
    fit_mask=folds!=fold;check_mask=~fit_mask
    fitted_bias=fit_biases(probabilities[fit_mask],labels[fit_mask])
    checked=predict_with_bias(probabilities[check_mask],fitted_bias)
    raw=raw_oof_pred[check_mask]
    meta_rows.append({'fold':fold,'raw_macro_F1':f1_score(labels[check_mask],raw,average='macro'),
        'biased_macro_F1':f1_score(labels[check_mask],checked,average='macro'),
        'biases':fitted_bias.tolist()})
meta=pd.DataFrame(meta_rows)
display(meta)
print('Mean paired meta-fold gain:',(meta.biased_macro_F1-meta.raw_macro_F1).mean())
best_bias=fit_biases(probabilities,labels)
oof_adjusted=predict_with_bias(probabilities,best_bias)
print('Selected bias:',best_bias.tolist())
print('Same-OOF tuned macro F1:',f1_score(labels,oof_adjusted,average='macro'))
print('OOF per-class F1 raw:',f1_score(labels,raw_oof_pred,average=None,labels=classes))
print('OOF per-class F1 biased:',f1_score(labels,oof_adjusted,average=None,labels=classes))

,fold,raw_macro_F1,biased_macro_F1,biases
0,1,0.756123,0.755603,"[0.0, 0.15, 0.25, 0.3]"
1,2,0.746062,0.746668,"[0.0, -0.0, 0.2, 0.2]"
2,3,0.750865,0.751398,"[0.0, 0.15, 0.25, 0.45]"
3,4,0.758046,0.757135,"[0.0, 0.15, 0.3, 0.5]"
4,5,0.751726,0.753036,"[0.0, 0.15, 0.3, 0.5]"


Mean paired meta-fold gain: 0.00020386751287089223


Selected bias: [0.0, 0.15, 0.3, 0.5]
Same-OOF tuned macro F1: 0.7542480179839841
OOF per-class F1 raw: [0.82147788 0.74119902 0.70999658 0.7376068 ]
OOF per-class F1 biased: [0.81897861 0.74298318 0.71026334 0.74476694]


In [4]:
name='t2_cat_class_bias_cv5_v3'
SAVE=SRC/'task2'/'save'
model_path,card_path,eval_path=(SAVE/f'{name}.joblib',SAVE/f'{name}.json',SAVE/f'{name}-evaluation.json')
if not (model_path.exists() and card_path.exists() and eval_path.exists()):
    assert not any(path.exists() for path in (model_path,card_path,eval_path)),'Inspect incomplete artifact'
    adjusted=ClassBiasClassifier(baseline,best_bias,categorical)
    adjusted_pred=adjusted.predict(X_valid)
    save_model(task=2,name=name,model=adjusted,data_name=DATA_NAME,
        card={'idea':'Development-OOF class log-probability biases on saved CatBoost',
              'base_model':model_name,'biases':best_bias.tolist(),
              'selection':'five development folds; class 0 reference bias fixed at zero',
              'seed':SEED,'meta_fold_mean_gain':float((meta.biased_macro_F1-meta.raw_macro_F1).mean())})
    result=evaluate_model(task=2,name=name,y_true=y_valid,y_pred=adjusted_pred,
        card={'split':'original stratified 20% holdout','seed':SEED,
              'biases_selected_on':'development OOF only'})
    print(name,result['score'])
else:
    adjusted=joblib.load(model_path);adjusted_pred=adjusted.predict(X_valid)
    print(name,json.loads(eval_path.read_text(encoding='utf8'))['score'])
print('Holdout macro F1 delta:',f1_score(y_valid,adjusted_pred,average='macro')-baseline_score)
print('Adjusted per-class report:');print(classification_report(y_valid,adjusted_pred,digits=4,zero_division=0))
print('Changed holdout predictions:',int((adjusted_pred!=baseline_pred).sum()))

t2_cat_class_bias_cv5_v3 0.7522162958586714
Holdout macro F1 delta: 0.0018018746595525048
Adjusted per-class report:
              precision    recall  f1-score   support

           0     0.8026    0.8310    0.8166      8191
           1     0.7513    0.7453    0.7483      5207
           2     0.7283    0.6845    0.7057      4206
           3     0.7378    0.7387    0.7383      3196

    accuracy                         0.7658     20800
   macro avg     0.7550    0.7499    0.7522     20800
weighted avg     0.7648    0.7658    0.7650     20800

Changed holdout predictions: 935


## 3. Bias-range sensitivity on development folds

The selected class-3 offset touches the first search boundary. Repeat the nested fold check with a wider range on the cached development probabilities. Do not use the holdout to choose between ranges.

In [5]:
def fit_wider_biases(probability,actual):
    logp=np.log(np.clip(probability,1e-12,1))
    bias=np.zeros(4,dtype=float)
    grid=np.round(np.arange(-1.0,1.001,.10),2)
    for _ in range(3):
        changed=False
        for label in (1,2,3):
            best=bias[label];best_score=-1.0
            for candidate in grid:
                trial=bias.copy();trial[label]=candidate
                score=f1_score(actual,classes[np.argmax(logp+trial,axis=1)],average='macro')
                if score>best_score+1e-10 or (abs(score-best_score)<=1e-10 and abs(candidate)<abs(best)):
                    best_score,best=float(score),float(candidate)
            if best!=bias[label]:changed=True
            bias[label]=best
        if not changed:break
    return bias
wider=[]
for fold in range(1,6):
    fit=folds!=fold;check=~fit
    bias=fit_wider_biases(probabilities[fit],labels[fit])
    wider.append({'fold':fold,'biases':bias.tolist(),
        'macro_F1':f1_score(labels[check],predict_with_bias(probabilities[check],bias),average='macro')})
wider=pd.DataFrame(wider)
display(wider)
print('Wider minus narrow paired mean F1:',(wider.macro_F1-meta.biased_macro_F1).mean())
print('Wider minus raw paired mean F1:',(wider.macro_F1-meta.raw_macro_F1).mean())
print('Wider beats narrow folds:',int((wider.macro_F1>meta.biased_macro_F1).sum()))

,fold,biases,macro_F1
0,1,"[0.0, 0.1, 0.3, 0.3]",0.756234
1,2,"[0.0, -0.0, 0.2, 0.2]",0.746668
2,3,"[0.0, 0.1, 0.3, 0.4]",0.751994
3,4,"[0.0, 0.1, 0.2, 0.4]",0.756905
4,5,"[0.0, 0.1, 0.3, 0.4]",0.753116


Wider minus narrow paired mean F1: 0.00021549901106101554
Wider minus raw paired mean F1: 0.00041936652393190774
Wider beats narrow folds: 3
